## Instrucciones realizar lambda ECR

### Introducción

estas instrucciones nos indican como se debe hacer para hacer deploy del proyecto a Lambda por medio de una imagen docker

### Requisitos
* Docker desktop
* Python
* AWS CLI
* Permisos respectivos en IAM 

**nota:** en este caso se ha creado el Lambda en la region **us-east-2**

### FLUJO COMPLETO: DOCKER -> ECR -> LAMBDA

### carpeta lista con archivos

se debe tener una carpeta lista con los archivos a usar

* lambda_function.py
* bedrock_client.py
* predict.py
* Dockerfile
* requirements.txt

In [ ]:
# dockerfile

FROM public.ecr.aws/lambda/python:3.11

COPY requirements.txt .

RUN pip install --upgrade pip

RUN pip install --only-binary=:all: -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

COPY app.py ${LAMBDA_TASK_ROOT}
COPY bedrock_client.py ${LAMBDA_TASK_ROOT}
COPY predict.py ${LAMBDA_TASK_ROOT}

CMD ["app.lambda_handler"]

Estando en la carpeta y con esos archivos , abrimos el terminal y desde ahi empezamos a ejecutar los pasos

### 1. Construir imagen Docker
```cmd 
docker build --platform linux/amd64 --no-cache -t chatbot-lambda . 
```
POR QUÉ: crea la imagen compatible con AWS Lambda (linux/amd64) y evita errores de cache.

### 2. Verificar imagen
```cmd 
docker images 
```
POR QUÉ: confirma que la imagen fue creada correctamente.

### 3. Tag para ECR
```cmd 
docker tag chatbot-lambda:latest <id_account>.dkr.ecr.us-east-2.amazonaws.com/chatbot-lambda:latest 
```
POR QUÉ: asocia la imagen local con el repositorio de AWS ECR.

### 4. Login a ECR
```cmd 
aws ecr get-login-password --region us-east-2 | docker login --username AWS --password-stdin <id_account>.dkr.ecr.us-east-2.amazonaws.com 
```
POR QUÉ: autentica Docker con AWS para permitir push.

### 5. Subir imagen a ECR
```cmd 
docker push <id_account>.dkr.ecr.us-east-2.amazonaws.com/chatbot-lambda:latest
```
POR QUÉ: sube la imagen a AWS para que Lambda la pueda usar.

### 6. (OPTIMIZADO) Build + Push en un paso
```cmd 
docker buildx build --platform linux/amd64 --no-cache -t <id_account>.dkr.ecr.us-east-2.amazonaws.com/chatbot-lambda:latest --push . 
```
POR QUÉ: construye y sube en un solo paso y evita errores de formato de imagen.

### 7. Crear Lambda (AWS Console)
- Tipo: Container Image
- URI: <id_account>.dkr.ecr.us-east-2.amazonaws.com/chatbot-lambda:latest
POR QUÉ: conecta Lambda con la imagen en ECR.

### 8. Probar Lambda
teniendo ya creado el lamba se procede a hacer pruebas.
Recordar siempre configurar, timeout, capacidad, test, colocar las variables de entorno que se usan en los scripts

* MY_REG_AWS
* KB_ID
* DATA_SOURCE_ID
* BEDROCK_ROLE_ARN
* INFERENCE_PROFILE_ARN

